![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Autoencoders & Clustering — Finding Structure Without Labels

---

So far, every model we've trained needed **labels** — someone had to tell it "this is a cat," "this is spam."

Today we do something different: we give the model **only images**, no labels at all, and ask two questions:

1. **Can it learn to compress and rebuild an image on its own?** → this is an **Autoencoder**
2. **Can it discover that some images naturally belong together — like all the "0"s, all the "1"s — without ever being told?** → this is **Clustering**

Both parts use the *same* trained model. The autoencoder learns to squeeze each digit down into just **2 numbers**. Then, in Part 2, we treat those 2 numbers as a point on a map and ask a clustering algorithm to find groups — with zero access to the real digit labels.

---
## Roadmap

```
Part 1 — Autoencoder
  Step 1  Setup
  Step 2  Load the data (images only, no labels used)
  Step 3  Understand the Autoencoder architecture
  Step 4  Build the model
  Step 5  Train it
  Step 6  Check the reconstructions

Part 2 — Clustering the Latent Space
  Step 7   Extract the compressed codes
  Step 8   Plot the codes — can we see structure?
  Step 9   Cluster the codes with K-Means
  Step 10  Compare clusters to the real digit labels
  Step 11  Bonus: generate brand new digits
```


---
## Step 1 — Setup

Standard imports. Nothing new here — `torch` for the model, `matplotlib` for pictures,
and `sklearn`'s `KMeans` for Part 2's clustering step.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
from torchvision.datasets import MNIST

import matplotlib.pyplot as plt
import numpy as np

from sklearn.cluster import KMeans

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## Step 2 — Load the Data

We use **MNIST**: 70,000 small grayscale images of handwritten digits (0–9), each 28×28 pixels.

Normally we'd also load the labels (which digit each image is). This time we **deliberately ignore them**
during training — the autoencoder will never see "this is a 7." It only ever sees the raw pixels.

We'll keep the labels around quietly in the background, only to check our results at the very end in Part 2.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),   # converts pixel values from [0, 255] to [0.0, 1.0]
])

train_dataset = MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset  = MNIST(root="./data", train=False, transform=transform, download=True)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Test images:     {len(test_dataset)}")

In [ ]:
# --- Look at a few sample digits ---
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"label: {labels[i].item()}", fontsize=10)
    ax.axis("off")

fig.suptitle("Sample MNIST Digits", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Image shape: {images[0].shape}   (1 channel, 28x28 pixels)")

---
## Step 3 — What is an Autoencoder?

An **autoencoder** is a neural network with a very simple job: **take an image in, produce the same image back out.**

That sounds pointless at first — why train a model to copy its input? The trick is *how* we make it copy:

```
  Input Image          Compressed Code           Reconstructed Image
  (28 x 28)      →         (just 2               →      (28 x 28)
  784 numbers            numbers!)                     784 numbers
                    ↑
              the "bottleneck" —
           forces the model to keep
             ONLY the essentials
```

The network is split into two halves:

| Part | Job | Also called |
|---|---|---|
| **Encoder** | Squeezes the image down into a tiny code | "compression" |
| **Decoder** | Expands that tiny code back into a full image | "reconstruction" |

Because the code in the middle is so small (**only 2 numbers**, compared to 784 pixels), the model is
**forced** to throw away anything unimportant and keep only the core shape of the digit.

That squeezed-down representation is called the **latent space** — and it's the whole reason this lab is interesting.
Once every digit is described by just 2 numbers, we can plot it, compare it, and — in Part 2 — cluster it.

---
## Step 4 — Build the Autoencoder

Our encoder uses convolutional layers (good at understanding images) to shrink the 28×28 image down step by step,
ending in a `Linear` layer that produces exactly **2 numbers**.

The decoder does the exact reverse: starting from those 2 numbers, it expands back up to a full 28×28 image.

```
ENCODER                                DECODER
28x28 image                            2 numbers
   |  Conv (stride 2)                     |  Linear
14x14                                  7x7 feature map
   |  Conv (stride 2)                     |  ConvTranspose (stride 2)
7x7                                    14x14
   |  Flatten + Linear                    |  ConvTranspose (stride 2)
2 numbers  <- the "code"               28x28 image
```

`ConvTranspose2d` is the reverse of `Conv2d` — instead of shrinking an image, it grows one back up.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()

        # ENCODER: image -> small code
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),   # 28x28 -> 14x14
            nn.BatchNorm2d(16),
            nn.LeakyReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),  # 14x14 -> 7x7
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, latent_dim),   # squeeze down to just `latent_dim` numbers
        )

        # DECODER: small code -> image
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 7 * 7),
            nn.LeakyReLU(),
            nn.Unflatten(1, (32, 7, 7)),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # 7x7 -> 14x14
            nn.BatchNorm2d(16),
            nn.LeakyReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),   # 14x14 -> 28x28
            nn.Sigmoid(),   # squashes pixel values back into [0, 1]
        )

    def forward(self, x):
        code = self.encoder(x)
        reconstruction = self.decoder(code)
        return code, reconstruction

LATENT_DIM = 2   # deliberately tiny, so we can plot the latent space directly in Part 2

model = Autoencoder(latent_dim=LATENT_DIM).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

---
## Step 5 — Train the Autoencoder

There's no "correct label" to predict here — the target is simply **the input image itself**.
We measure how different the reconstruction is from the original using **MSE (Mean Squared Error)**:
the average of `(original_pixel - reconstructed_pixel)^2` across every pixel.

Training pushes this error down, which means the reconstructions get closer and closer to the originals.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 12
train_losses = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, _ in train_loader:          # the "_" means we ignore the labels
        images = images.to(device)

        _, reconstructions = model(images)
        loss = criterion(reconstructions, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]  Reconstruction Loss: {avg_loss:.4f}")

In [ ]:
# --- Plot how the loss dropped over training ---
plt.figure(figsize=(7, 3.5))
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, 'b-o')
plt.xlabel("Epoch")
plt.ylabel("Reconstruction Loss (MSE)")
plt.title("Training Progress")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 6 — Check the Reconstructions

Let's see how well the model can rebuild digits it has seen, after squeezing them down to just 2 numbers.

Expect the reconstructions to look a bit **blurry or imperfect** — that's normal and expected here.
Compressing an entire digit into only 2 numbers is extreme; a little quality loss is the trade-off we accepted
on purpose, in exchange for being able to see the latent space directly in Part 2.

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images = images[:10].to(device)

with torch.no_grad():
    _, reconstructions = model(images)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for i in range(10):
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstructions[i].cpu().squeeze(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Rebuilt", fontsize=11)
fig.suptitle("Original (top) vs. Reconstructed from 2 Numbers (bottom)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Part 2 — Clustering the Latent Space

The autoencoder just learned to describe every digit using only **2 numbers** — without ever being told which digit it was looking at.

Now for the interesting question: **do those 2 numbers actually mean anything?**

If the model organized similar-looking digits near each other in that 2-number space, we should be able to
find natural groups in it — purely by looking at the numbers, with **no labels involved at all.**
That's exactly what clustering does.

---
## Step 7 — Extract the Compressed Codes

We run every test image through the **encoder only** (not the decoder) and save the resulting 2 numbers for each image.
We also keep the true labels off to the side — but only for checking our work at the end, never for the clustering itself.

In [ ]:
model.eval()

all_codes = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        codes, _ = model(images)          # encoder output only
        all_codes.append(codes.cpu())
        all_labels.append(labels)

all_codes  = torch.cat(all_codes).numpy()   # shape: (num_images, 2)
all_labels = torch.cat(all_labels).numpy()  # shape: (num_images,)  -- kept aside, not used yet

print(f"Latent codes shape: {all_codes.shape}")
print(f"Each image is now just {all_codes.shape[1]} numbers, e.g.: {all_codes[0]}")

---
## Step 8 — Plot the Codes: Can We See Structure?

Since every image is now just an (x, y) point, we can plot all 10,000 test images on a simple 2D graph.

We plot them all in a single color first — **on purpose, no labels yet** — to ask honestly:
does this look like random scattered noise, or do you see natural groupings forming?

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(all_codes[:, 0], all_codes[:, 1], s=4, alpha=0.4, color="gray")
plt.xlabel("Latent dimension 1")
plt.ylabel("Latent dimension 2")
plt.title("The Latent Space (no labels used)")
plt.tight_layout()
plt.show()

---
## Step 9 — Cluster the Codes with K-Means

**K-Means** is one of the simplest clustering algorithms. It finds groups by repeating 3 steps:

```
1. PLACE CENTERS     Drop K random points onto the map
2. ASSIGN            Every point joins its nearest center
3. MOVE & REPEAT      Each center moves to the middle of its group, then repeat step 2
```

It keeps repeating "assign, then move" until the groups stop changing. We choose `K` — the number of groups
to look for. Since we know MNIST has 10 digits, we'll ask for **10 clusters** — but note that K-Means
itself has no idea what a "digit" is. It only sees the 2 numbers per image and groups nearby points together.

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
cluster_assignments = kmeans.fit_predict(all_codes)

print(f"Found {len(np.unique(cluster_assignments))} clusters")
print(f"First 10 images were assigned to clusters: {cluster_assignments[:10]}")

---
## Step 10 — Compare Clusters to the Real Digit Labels

Now for the reveal. We plot the same latent space twice, side by side:

- **Left:** colored by the clusters K-Means found — using only the 2 numbers, no labels
- **Right:** colored by the *real* digit labels — which K-Means never saw

If the autoencoder's latent space captured meaningful structure, the two pictures should look similar:
groups that K-Means found on the left should roughly line up with real digits on the right.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cmap = plt.cm.tab10
colors = [cmap(i) for i in range(10)]

# --- Left: K-Means clusters ---
axes[0].scatter(all_codes[:, 0], all_codes[:, 1], c=cluster_assignments,
                 cmap=cmap, s=4, alpha=0.5, vmin=0, vmax=9)
axes[0].set_title("Colored by K-Means Cluster\n(no labels used)", fontsize=12)
axes[0].set_xlabel("Latent dimension 1")
axes[0].set_ylabel("Latent dimension 2")

cluster_handles = [mpatches.Patch(color=colors[i], label=f"Cluster {i}") for i in range(10)]
axes[0].legend(handles=cluster_handles, bbox_to_anchor=(1.02, 1), loc="upper left",
                fontsize=8, title="K-Means Group", title_fontsize=9, frameon=False)

# --- Right: real digit labels ---
axes[1].scatter(all_codes[:, 0], all_codes[:, 1], c=all_labels,
                 cmap=cmap, s=4, alpha=0.5, vmin=0, vmax=9)
axes[1].set_title("Colored by Real Digit Label\n(revealed only now, for comparison)", fontsize=12)
axes[1].set_xlabel("Latent dimension 1")
axes[1].set_ylabel("Latent dimension 2")

digit_handles = [mpatches.Patch(color=colors[i], label=f"Digit {i}") for i in range(10)]
axes[1].legend(handles=digit_handles, bbox_to_anchor=(1.02, 1), loc="upper left",
                fontsize=8, title="True Digit", title_fontsize=9, frameon=False)

plt.tight_layout()
plt.show()

**What to notice:** the colors won't match perfectly (K-Means has no idea "this blob is a 3" — it only
knows "these points are close together"), but the *shapes* of the groups should look strikingly similar between
the two plots. That similarity is the whole point: **the autoencoder discovered meaningful structure in the data,
purely by learning to compress and rebuild images — with zero access to labels.**

This is the essence of unsupervised learning: letting the data speak for itself.

---
## Step 11 — Bonus: Generate Brand New Digits

Since the decoder can turn *any* 2 numbers into an image, we can pick random points from the latent space
and see what digit comes out — even points the model never saw during training.

This won't produce perfect digits (that's more the territory of the VAEs and diffusion models from today's
lecture), but it's a fun preview of the same core idea: **sample a point from the latent space, decode it,
get something new.**

In [ ]:
model.eval()

# Pick random points within the range the real data occupies
x_min, x_max = all_codes[:, 0].min(), all_codes[:, 0].max()
y_min, y_max = all_codes[:, 1].min(), all_codes[:, 1].max()

random_points = np.column_stack([
    np.random.uniform(x_min, x_max, 10),
    np.random.uniform(y_min, y_max, 10),
])
random_points = torch.tensor(random_points, dtype=torch.float32).to(device)

with torch.no_grad():
    generated = model.decoder(random_points).cpu()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(generated[i].squeeze(), cmap="gray")
    ax.axis("off")

fig.suptitle("Digits Generated from Random Latent Points", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Conclusion

| Step | What we did | Key idea |
|---|---|---|
| Build the autoencoder | Encoder compresses, decoder rebuilds | A tiny bottleneck forces the model to keep only what matters |
| Train it | Minimize reconstruction error (MSE) | No labels needed — the input is its own target |
| Extract the latent codes | Run the encoder only | Every image becomes just 2 numbers |
| Cluster with K-Means | Group nearby points | Structure can be found with zero labels |
| Compare to real labels | Side-by-side plot | The discovered groups lined up with real digits |
| Generate new digits | Decode random points | Sampling the latent space creates new data |

**The big takeaway:** an autoencoder doesn't just compress images — it learns a *meaningful map* of the data.
Nearby points in that map tend to be similar images. That's exactly why clustering worked on it, and it's the
same core idea behind every generative model in today's lecture: **learn a good latent space, then sample from it.**

---



#### **Contributed by Hussain Alyafei**